# GitHub User Churn Prediction
## Introduction to Data Science — Final Project

**Student:** Cesar Oyola  
**Project:** Customer Churn Predictor App  
**Data source:** GitHub REST API  
**Goal:** predict whether a GitHub user is likely to churn, using public activity data.

This notebook is the analysis and experimentation part of the project. The production code is stored in the `app/` folder:

- `scraper.py` collects raw GitHub data and creates the churn label.
- `features.py` generates the machine-learning feature dataset.
- `model.py` trains and saves the final model using the selected features.
- `main.py` exposes the trained model through a FastAPI `/predict` endpoint.

## Project Objective

The objective is to build a Dockerized web application that predicts user churn. In this project, churn is simulated using GitHub inactivity.

The workflow followed in this notebook and codebase is:

1. **Data Collection** — collect public user, repository, and event data from the GitHub API.
2. **Churn Labeling** — label a user as churned when inactivity is greater than 180 days.
3. **Feature Engineering** — transform raw API fields into model-ready features.
4. **Feature Selection** — apply four methods: Filter, RFE, Decision Tree, and Random Forest.
5. **Final Model Selection** — choose the final features and use them in `model.py`.
6. **Deployment** — expose the model using FastAPI and Docker.
7. **Retention Analysis** — interpret the results as business/user-retention insights.

## Imports

The notebook uses the same main libraries as the app: `pandas`, `numpy`, `scikit-learn`, and `matplotlib`.

In [ ]:
from pathlib import Path
import ast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_selection import f_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
TOP_K = 8

## Helper Functions for Project Paths

The notebook may be opened from the project root or from the `notebooks/` folder. These helper functions locate the project files safely.

In [ ]:
def find_project_root() -> Path:
    """
    Locate the project root by looking for app/ and data/ folders.
    Works when the notebook is run from the project root or notebooks/.
    """
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]

    for candidate in candidates:
        if (candidate / "app").exists() and (candidate / "data").exists():
            return candidate

    return Path.cwd()


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
APP_DIR = PROJECT_ROOT / "app"

FEATURE_PATH = PROCESSED_DIR / "github_features.csv"
FEATURE_METADATA_PATH = PROCESSED_DIR / "github_feature_metadata.csv"
MODEL_PY_PATH = APP_DIR / "model.py"

print("Project root:", PROJECT_ROOT)
print("Feature dataset:", FEATURE_PATH)
print("Model.py:", MODEL_PY_PATH)

# 1. Data Collection

The project uses the **GitHub REST API**. The final `scraper.py` collects three categories of public data:

### User profile data
Examples: username, account creation date, last profile update, public repositories, followers, following, public gists, bio, company, and location.

### Repository data
Examples: repository name, creation date, last push date, stars, forks, size, language, archived status, fork status, and open issues.

### Public event data
Examples: event type, event creation date, repository name, and commit count for `PushEvent` records.

The raw data is saved into:

- `data/raw/github_users_raw.csv`
- `data/raw/github_repos_raw.csv`
- `data/raw/github_events_raw.csv`
- `data/raw/github_users_labeled.csv`

In [ ]:
raw_files = [
    RAW_DIR / "github_users_raw.csv",
    RAW_DIR / "github_repos_raw.csv",
    RAW_DIR / "github_events_raw.csv",
    RAW_DIR / "github_users_labeled.csv",
]

for file_path in raw_files:
    if file_path.exists():
        temp_df = pd.read_csv(file_path)
        print(f"{file_path.name}: {temp_df.shape[0]} rows, {temp_df.shape[1]} columns")
    else:
        print(f"Missing: {file_path}")

# 2. Churn Label Definition

The GitHub API does not provide a direct `churned` column, so the project defines churn using inactivity.

The rule used in `scraper.py` is:

```text
churned = 1 if days_since_last_activity > 180
churned = 0 otherwise
```

The `last_activity_at` value is created using the latest available activity date from:

- the user profile update date,
- the latest repository push date,
- the latest public event date.

The 180-day threshold is used because GitHub users can naturally pause public activity for weeks or months. Six months of inactivity is a reasonable signal of disengagement for this project. However, this is still a **simulated churn label**, not a confirmed account cancellation.

# 3. Load the Feature Dataset

The file `github_features.csv` is created by running:

```bash
docker-compose run --rm churn-api python features.py
```

This dataset contains one row per user, the generated features, and the target label `churned`.

In [ ]:
if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {FEATURE_PATH}. Run Step 3 and Step 4 first."
    )

df = pd.read_csv(FEATURE_PATH)

print("Dataset shape:", df.shape)
df.head()

In [ ]:
print("Columns in feature dataset:")
for column in df.columns:
    print("-", column)

## Class Balance

Before training or selecting features, it is important to check the balance between active and churned users. A very imbalanced dataset can make the model biased toward the majority class.

In [ ]:
class_counts = df["churned"].value_counts().sort_index()
class_percentages = df["churned"].value_counts(normalize=True).sort_index() * 100

balance_df = pd.DataFrame({
    "count": class_counts,
    "percentage": class_percentages.round(2),
})

balance_df.index = balance_df.index.map({0: "active / retained", 1: "churned"})
balance_df

In [ ]:
plt.figure(figsize=(6, 4))
class_counts.plot(kind="bar")
plt.title("Class Balance: Active vs Churned Users")
plt.xlabel("Churn Label")
plt.ylabel("Number of Users")
plt.xticks(ticks=[0, 1], labels=["Active", "Churned"], rotation=0)
plt.show()

# 4. Feature Engineering Summary

The project requires generated features, not just raw API fields. The final `features.py` creates 28 candidate features divided into four categories:

1. **Time-based features** — recency and account age.
2. **Ratio/proportion features** — normalized activity or engagement rates.
3. **Aggregation features** — summarized repository and event history.
4. **Binary features** — presence or absence of important user attributes.

This satisfies the requirement of having at least 8 meaningful features.

In [ ]:
if FEATURE_METADATA_PATH.exists():
    feature_metadata = pd.read_csv(FEATURE_METADATA_PATH)
else:
    feature_metadata = pd.DataFrame({
        "feature": [c for c in df.columns if c not in ["username", "churned"]],
        "type": "generated feature",
        "reason": "Generated by features.py",
    })

feature_metadata

In [ ]:
feature_type_summary = (
    feature_metadata
    .groupby("type")
    .size()
    .reset_index(name="number_of_features")
)

feature_type_summary

# 5. Prepare Data for Feature Selection

For feature selection:

- `X` contains the generated features.
- `y` contains the target label `churned`.
- `username` is removed because it is only an identifier, not a predictive feature.

In [ ]:
if "churned" not in df.columns:
    raise ValueError("The dataset must contain a churned column.")

columns_to_drop = ["churned"]
if "username" in df.columns:
    columns_to_drop.append("username")

X = df.drop(columns=columns_to_drop)
y = df["churned"]

X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
y = pd.to_numeric(y, errors="coerce").fillna(0).astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target classes:", sorted(y.unique()))

if y.nunique() < 2:
    raise ValueError(
        "The churned target has only one class. Feature selection needs both 0 and 1 labels."
    )

# 6. Feature Selection Method 1 — Filter Methods

Filter methods evaluate features before training the final model. This notebook combines:

- **ANOVA F-test:** measures how strongly each feature differs between churned and active users.
- **Variance check:** penalizes features that barely change.
- **Correlation check:** penalizes features that are highly redundant with another feature.

Filter methods are fast and useful for an initial ranking, but they evaluate features mostly independently.

In [ ]:
def run_filter_methods(X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    """
    Method 1: Filter methods.
    Combines ANOVA F-test, low variance detection, and high-correlation penalty.
    """
    f_scores, p_values = f_classif(X, y)

    filter_df = pd.DataFrame({
        "Feature": X.columns,
        "anova_score": np.nan_to_num(f_scores, nan=0.0, posinf=0.0, neginf=0.0),
        "p_value": np.nan_to_num(p_values, nan=1.0, posinf=1.0, neginf=1.0),
    })

    variances = X.var()
    filter_df["variance"] = filter_df["Feature"].map(variances)
    filter_df["low_variance"] = filter_df["variance"] <= 0.01

    corr_matrix = X.corr().abs()
    upper_triangle = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    redundant_features = set()
    anova_by_feature = filter_df.set_index("Feature")["anova_score"]

    for column in upper_triangle.columns:
        correlated_features = upper_triangle.index[upper_triangle[column] > 0.90].tolist()

        for other_column in correlated_features:
            if anova_by_feature[column] >= anova_by_feature[other_column]:
                redundant_features.add(other_column)
            else:
                redundant_features.add(column)

    filter_df["high_correlation_penalty"] = filter_df["Feature"].isin(redundant_features)

    filter_df["filter_score"] = filter_df["anova_score"]
    filter_df.loc[filter_df["low_variance"], "filter_score"] *= 0.25
    filter_df.loc[filter_df["high_correlation_penalty"], "filter_score"] *= 0.50

    filter_df["Filter Rank"] = (
        filter_df["filter_score"]
        .rank(ascending=False, method="min")
        .astype(int)
    )

    return filter_df.sort_values("Filter Rank").reset_index(drop=True)

In [ ]:
filter_df = run_filter_methods(X, y)
filter_df.head(10)

# 7. Feature Selection Method 2 — Recursive Feature Elimination (RFE)

Recursive Feature Elimination is a wrapper method. It trains a model, removes weaker features, and repeats the process until only the selected number of features remains.

Here, Logistic Regression is used as the estimator. Because Logistic Regression is sensitive to scale, the features are standardized before applying RFE.

In [ ]:
def run_rfe_method(X: pd.DataFrame, y: pd.Series, top_k: int = TOP_K) -> pd.DataFrame:
    """
    Method 2: Recursive Feature Elimination using Logistic Regression.
    """
    n_features_to_select = min(top_k, X.shape[1])

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    estimator = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )

    rfe = RFE(estimator=estimator, n_features_to_select=n_features_to_select)
    rfe.fit(X_scaled, y)

    rfe_df = pd.DataFrame({
        "Feature": X.columns,
        "RFE Selected": rfe.support_,
        "RFE Rank": rfe.ranking_,
    })

    return rfe_df.sort_values(["RFE Rank", "Feature"]).reset_index(drop=True)

In [ ]:
rfe_df = run_rfe_method(X, y, top_k=TOP_K)
rfe_df.head(15)

# 8. Feature Selection Method 3 — Decision Tree Importance

A Decision Tree ranks features based on how much they reduce impurity when splitting the data into active and churned users.

This method is interpretable, but a single tree can be unstable because small changes in the dataset may change the ranking.

In [ ]:
def run_decision_tree_method(X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    """
    Method 3: Feature importance from a single Decision Tree.
    """
    tree = DecisionTreeClassifier(
        max_depth=5,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )

    tree.fit(X, y)

    dt_df = pd.DataFrame({
        "Feature": X.columns,
        "dt_importance": tree.feature_importances_,
    })

    dt_df["DT Rank"] = (
        dt_df["dt_importance"]
        .rank(ascending=False, method="min")
        .astype(int)
    )

    return dt_df.sort_values("DT Rank").reset_index(drop=True)

In [ ]:
dt_df = run_decision_tree_method(X, y)
dt_df.head(10)

In [ ]:
plt.figure(figsize=(8, 5))
plot_df = dt_df.head(10).sort_values("dt_importance")
plt.barh(plot_df["Feature"], plot_df["dt_importance"])
plt.title("Top Decision Tree Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

# 9. Feature Selection Method 4 — Random Forest Importance

Random Forest trains many decision trees and averages their feature importances. This usually produces a more stable ranking than a single Decision Tree.

In [ ]:
def run_random_forest_method(X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    """
    Method 4: Feature importance from a Random Forest model.
    """
    forest = RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )

    forest.fit(X, y)

    rf_df = pd.DataFrame({
        "Feature": X.columns,
        "rf_importance": forest.feature_importances_,
    })

    rf_df["RF Rank"] = (
        rf_df["rf_importance"]
        .rank(ascending=False, method="min")
        .astype(int)
    )

    return rf_df.sort_values("RF Rank").reset_index(drop=True)

In [ ]:
rf_df = run_random_forest_method(X, y)
rf_df.head(10)

In [ ]:
plt.figure(figsize=(8, 5))
plot_df = rf_df.head(10).sort_values("rf_importance")
plt.barh(plot_df["Feature"], plot_df["rf_importance"])
plt.title("Top Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

# 10. Comparison of All Four Feature Selection Methods

The required final output of Step 5 is a comparison table. A feature is marked:

- **Keep** if it receives support from at least 3 of the 4 methods.
- **Optional** if it receives support from 2 methods.
- **Drop** if it receives support from 0 or 1 method.

This table is the main evidence used to justify the final selected features.

In [ ]:
def create_comparison_table(
    filter_df: pd.DataFrame,
    rfe_df: pd.DataFrame,
    dt_df: pd.DataFrame,
    rf_df: pd.DataFrame,
    top_k: int = TOP_K,
) -> pd.DataFrame:
    """
    Combine the results of the four feature selection methods.
    """
    comparison = (
        filter_df[["Feature", "Filter Rank"]]
        .merge(rfe_df[["Feature", "RFE Selected", "RFE Rank"]], on="Feature")
        .merge(dt_df[["Feature", "DT Rank"]], on="Feature")
        .merge(rf_df[["Feature", "RF Rank"]], on="Feature")
    )

    comparison["Filter Vote"] = comparison["Filter Rank"] <= top_k
    comparison["RFE Vote"] = comparison["RFE Selected"]
    comparison["DT Vote"] = comparison["DT Rank"] <= top_k
    comparison["RF Vote"] = comparison["RF Rank"] <= top_k

    comparison["Total Votes"] = comparison[
        ["Filter Vote", "RFE Vote", "DT Vote", "RF Vote"]
    ].sum(axis=1)

    comparison["Average Rank"] = comparison[
        ["Filter Rank", "RFE Rank", "DT Rank", "RF Rank"]
    ].mean(axis=1)

    def decide(row):
        if row["Total Votes"] >= 3:
            return "✅ Keep"
        elif row["Total Votes"] == 2:
            return "⚠️ Optional"
        else:
            return "❌ Drop"

    comparison["Decision"] = comparison.apply(decide, axis=1)
    comparison["RFE Selected"] = comparison["RFE Selected"].map({True: "✅", False: "❌"})

    decision_order = {"✅ Keep": 0, "⚠️ Optional": 1, "❌ Drop": 2}
    comparison["Decision Order"] = comparison["Decision"].map(decision_order)

    comparison = comparison.sort_values(
        by=["Decision Order", "Total Votes", "Average Rank"],
        ascending=[True, False, True],
    )

    display_columns = [
        "Feature",
        "Filter Rank",
        "RFE Selected",
        "DT Rank",
        "RF Rank",
        "Decision",
        "Total Votes",
        "Average Rank",
    ]

    return comparison[display_columns].reset_index(drop=True)

In [ ]:
comparison_df = create_comparison_table(
    filter_df=filter_df,
    rfe_df=rfe_df,
    dt_df=dt_df,
    rf_df=rf_df,
    top_k=TOP_K,
)

comparison_df

In [ ]:
comparison_output_path = PROCESSED_DIR / "feature_selection_comparison.csv"
comparison_df.to_csv(comparison_output_path, index=False)
print("Saved comparison table to:", comparison_output_path)

## Selected Features from the Comparison Table

The code below extracts the features marked as **Keep**. These are the features that should be used by the final model.

In [ ]:
selected_from_table = comparison_df.loc[
    comparison_df["Decision"] == "✅ Keep",
    "Feature",
].tolist()

if len(selected_from_table) < 5:
    optional_features = comparison_df.loc[
        comparison_df["Decision"] == "⚠️ Optional",
        "Feature",
    ].tolist()
    selected_from_table.extend(optional_features)

selected_from_table = selected_from_table[:TOP_K]

print("Selected features from notebook comparison:")
print("SELECTED_FEATURES = [")
for feature in selected_from_table:
    print(f'    "{feature}",')
print("]")

# 11. Verification Against `model.py`

The final production model uses the selected features written in `app/model.py`. This cell reads `model.py` and checks whether it matches the notebook decision.

In [ ]:
def get_selected_features_from_model_py(model_py_path: Path) -> list[str]:
    """
    Read SELECTED_FEATURES from app/model.py without importing the module.
    This avoids loading model.pkl during notebook analysis.
    """
    if not model_py_path.exists():
        return []

    source = model_py_path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    for node in tree.body:
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id == "SELECTED_FEATURES":
                    return ast.literal_eval(node.value)

    return []


selected_from_model_py = get_selected_features_from_model_py(MODEL_PY_PATH)

print("Selected features currently in app/model.py:")
print("SELECTED_FEATURES = [")
for feature in selected_from_model_py:
    print(f'    "{feature}",')
print("]")

missing_in_table = [f for f in selected_from_model_py if f not in comparison_df["Feature"].tolist()]
if missing_in_table:
    print("Warning: these model.py features do not exist in the comparison table:", missing_in_table)
else:
    print("All model.py selected features exist in the generated feature dataset.")

In [ ]:
model_alignment_df = comparison_df.copy()
model_alignment_df["Used in model.py"] = model_alignment_df["Feature"].isin(selected_from_model_py)
model_alignment_df["Used in model.py"] = model_alignment_df["Used in model.py"].map({True: "✅", False: ""})

model_alignment_df.head(15)

# 12. Important Interpretation Note

`days_since_last_activity` is expected to be very important because the churn label itself is based on inactivity.

This is not necessarily wrong, because recency is a real churn signal. However, it should be mentioned in the report: the model is partly learning the same inactivity rule that was used to define churn.

For a stronger analysis, it is useful to also look at features such as `recent_event_count`, `language_count`, `inactive_repo_ratio`, `avg_commits_per_week`, or profile-completeness features because they provide additional behavioral context.

# 13. Optional Sensitivity Check Without `days_since_last_activity`

This optional section checks which features become important if the strongest direct inactivity feature is removed. This is useful for discussion, but the final app can still use the selected features from `model.py`.

In [ ]:
if "days_since_last_activity" in X.columns:
    X_without_direct_inactivity = X.drop(columns=["days_since_last_activity"])

    filter_df_no_direct = run_filter_methods(X_without_direct_inactivity, y)
    rfe_df_no_direct = run_rfe_method(X_without_direct_inactivity, y, top_k=TOP_K)
    dt_df_no_direct = run_decision_tree_method(X_without_direct_inactivity, y)
    rf_df_no_direct = run_random_forest_method(X_without_direct_inactivity, y)

    comparison_no_direct_df = create_comparison_table(
        filter_df=filter_df_no_direct,
        rfe_df=rfe_df_no_direct,
        dt_df=dt_df_no_direct,
        rf_df=rf_df_no_direct,
        top_k=TOP_K,
    )

    comparison_no_direct_df.head(15)
else:
    print("days_since_last_activity was not found in X.")

# 14. Final Model and API Summary

The final model is trained in `app/model.py` using a Random Forest classifier. The selected features are stored in the `SELECTED_FEATURES` list.

The model is saved as `app/model.pkl` using `joblib`. The FastAPI app in `app/main.py` loads this saved model and exposes:

- `GET /health` — confirms the API is running.
- `GET /features` — returns the selected features required by the model.
- `POST /predict` — returns `churned` and `churn_probability`.

The `/predict` endpoint expects the same fields used by `model.py`.

In [ ]:
example_input = {
    "days_since_last_activity": 300,
    "account_age_days": 2000,
    "language_count": 3,
    "has_bio": 1,
    "has_company": 0,
    "has_location": 1,
    "recent_event_count": 5,
}

example_input

To test the API after running Docker:

```bash
docker-compose up --build
```

Open:

```text
http://localhost:8000/docs
```

Then use the example JSON input above in the `/predict` endpoint.

# 15. Retention Analysis

The model is not only a technical classifier. It can also be interpreted as a retention tool.

If a user has a high `days_since_last_activity`, the platform could send a re-engagement message or recommend repositories related to their previous interests.

If `recent_event_count` is low, GitHub could suggest trending repositories, beginner-friendly issues, or communities related to the user’s languages.

If `language_count` is low, GitHub could recommend learning resources or projects in related languages to increase engagement.

If profile-completeness features such as `has_bio`, `has_company`, or `has_location` are missing, the platform could gently encourage the user to complete their profile. A more complete profile may help the user connect with other developers and become more involved in the community.

The important idea is that the model should not only say who may churn. It should also help decide what kind of intervention might reduce churn risk.

# 16. Ethics and Limitations

The churn label in this project is simulated. A user marked as churned is not necessarily a user who deleted their GitHub account or permanently abandoned the platform. The label only means that their public activity has been inactive for more than 180 days.

The dataset also uses only public GitHub activity. Users may be active in private repositories, but that activity is not visible in this dataset. Therefore, some users may appear inactive even though they are active privately.

Another limitation is dataset size. The current dataset was enough to build and test the full pipeline, but a larger dataset of 200–500 users would make feature selection and model evaluation more stable.

Predictions should be treated as probabilities, not facts. A high churn probability should trigger helpful and respectful retention actions, not aggressive or unfair treatment.

# 17. Conclusion

This project built a complete churn prediction pipeline using GitHub API data. The system collects public user, repository, and event data, creates a churn label based on inactivity, generates meaningful features, compares four feature selection methods, trains a Random Forest model, and deploys the model through a Dockerized FastAPI application.

The feature selection step showed which behavioral signals were most useful for the current dataset. The final app uses the selected features from `model.py` and returns a churn prediction and probability through `/predict`.

The main lesson is that feature engineering is central to machine learning. The model does not work directly from raw API fields; it depends on transformed features that represent meaningful engagement behavior.

# Final Checklist

Before submitting, verify that:

- `docker-compose up --build` starts the API.
- `http://localhost:8000/health` returns status OK.
- `http://localhost:8000/features` returns the selected features.
- `http://localhost:8000/docs` can test `/predict`.
- The notebook includes the four feature selection methods and comparison table.
- The report explains which features matter most and why.
- The limitations section mentions the small dataset and public-only GitHub data.